# Exploring the details of specific routes of the WVV

In [4]:
import pandas as pd

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

ROUTE_SHORT_NAME = "14"

### Extracting Route Sequences

In [5]:
df_stoptimes_wvv = pd.read_csv("../data_extract/stoptimes_wvv.csv", dtype={"route_short_name": str})
df_stoptimes_route = df_stoptimes_wvv.loc[df_stoptimes_wvv["route_short_name"] == ROUTE_SHORT_NAME]

df_stops_wvv = pd.read_csv("../data_extract/stops_wvv.csv")
df_stops_route = df_stoptimes_route.merge(df_stops_wvv, on="stop_id", how="inner")
df_stops_route = df_stops_route[["trip_id", "route_id", "route_short_name", "route_type", "arrival_time", "departure_time",
                                 "stop_id", "stop_name" , "parent_station", "stop_lat", "stop_lon", "platform_code", "stop_sequence"]]

df_route = df_stops_route.sort_values(by=["trip_id", "stop_sequence"])
trip_sequences = df_route.groupby("trip_id")["stop_id"].apply(tuple)
    
unique_sequences = {}
for trip_id, seq in trip_sequences.items():
    if seq not in unique_sequences:
        unique_sequences[seq] = trip_id
        
sequence_dfs = []
columns_to_keep = ["trip_id", "route_short_name", "stop_sequence", "stop_id", "stop_name", "stop_lat", "stop_lon", "platform_code", "parent_station"]

for seq, rep_trip_id in unique_sequences.items():
    seq_df = df_route[df_route["trip_id"] == rep_trip_id].copy()
    seq_df = seq_df[columns_to_keep]
    
    seq_df = seq_df.reset_index(drop=True)
    sequence_dfs.append(seq_df)


#for i, seq in enumerate(sequence_dfs, 1):
#    print(f"Sequence {i}: \n{seq}\n")


### Vizualizing the Sequences

In [6]:
from plot_sequences_map import plot_sequences_map

plot_sequences_map(sequence_dfs, f"sequence_map_{ROUTE_SHORT_NAME}")

Map saved as sequence_map_14.html. Open this file in your web browser!
